In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import random

# Set seed for reproducibility
np.random.seed(42)
random.seed(42)

# 1. Generate Mock Data (10,000 records)
num_records = 10000
currencies = ['USD', 'JPY', 'EUR', 'AUD', 'GBP', 'THB', 'CAD', 'SGD']
items = ['Mega Box', 'Gold Pass', 'Gem Pack (Small)', 'Gem Pack (Large)', 'Hero Skin', 'Resource Bundle']
exchange_rates = {
    'USD': 1.0,
    'JPY': 0.0067,
    'EUR': 1.08,
    'AUD': 0.65,
    'GBP': 1.27,
    'THB': 0.028,
    'CAD': 0.74,
    'SGD': 0.74
}

data = []
start_date = datetime(2024, 5, 1)
for i in range(num_records):
    sc_id = f"SC-{random.randint(100000, 999999)}"
    item = random.choice(items)
    currency = random.choice(currencies)
    
    # Base price in USD roughly
    base_price = random.choice([0.99, 4.99, 9.99, 19.99, 49.99, 99.99])
    # Convert to local currency (approximate)
    amount = round(base_price / exchange_rates[currency], 2)
    if currency == 'JPY':
        amount = round(amount, -1) # JPY usually doesn't have decimals

    # Random date within the month
    date = start_date + timedelta(days=random.randint(0, 30), hours=random.randint(0, 23))
    
    data.append({
        'transaction_id': f"TXN-{1000000 + i}",
        'supercell_id': sc_id,
        'item_name': item,
        'amount_local': amount,
        'currency': currency,
        'timestamp': date
    })

df = pd.DataFrame(data)

# 2. Processing (Month-End Closing Logic)
df['exchange_rate_to_usd'] = df['currency'].map(exchange_rates)
df['amount_usd'] = (df['amount_local'] * df['exchange_rate_to_usd']).round(2)
df['date'] = df['timestamp'].dt.date

# 3. Data Visualization for Monitoring
plt.style.use('ggplot')
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Supercell Month-End Closing Monitoring Dashboard', fontsize=20)

# Chart 1: Revenue by Item (USD)
revenue_by_item = df.groupby('item_name')['amount_usd'].sum().sort_values(ascending=False)
sns.barplot(x=revenue_by_item.values, y=revenue_by_item.index, ax=axes[0, 0], palette='viridis')
axes[0, 0].set_title('Total Revenue by Item (USD)')
axes[0, 0].set_xlabel('Revenue (USD)')

# Chart 2: Transaction Volume by Currency
tx_by_currency = df['currency'].value_counts()
axes[0, 1].pie(tx_by_currency, labels=tx_by_currency.index, autopct='%1.12f%%', startangle=140, colors=sns.color_palette('pastel'))
axes[0, 1].set_title('Transaction Volume Distribution by Currency')

# Chart 3: Daily Revenue Trend (USD)
daily_revenue = df.groupby('date')['amount_usd'].sum()
axes[1, 0].plot(daily_revenue.index, daily_revenue.values, marker='o', linestyle='-', color='tab:blue')
axes[1, 0].set_title('Daily Revenue Trend (USD)')
axes[1, 0].set_ylabel('Revenue (USD)')
plt.setp(axes[1, 0].xaxis.get_majorticklabels(), rotation=45)

# Chart 4: Revenue Contribution by Currency (USD)
rev_by_currency = df.groupby('currency')['amount_usd'].sum().sort_values(ascending=False)
sns.barplot(x=rev_by_currency.index, y=rev_by_currency.values, ax=axes[1, 1], palette='magma')
axes[1, 1].set_title('Revenue Contribution by Currency (USD)')
axes[1, 1].set_ylabel('Revenue (USD)')

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.savefig('month_end_monitoring_dashboard.png')

# Save closing report to CSV
df.to_csv('supercell_month_end_report.csv', index=False)

# Summary Stats for response
summary_stats = {
    'total_transactions': len(df),
    'total_revenue_usd': df['amount_usd'].sum(),
    'top_item': revenue_by_item.index[0],
    'top_currency': rev_by_currency.index[0]
}
print(summary_stats)